# Use your own data

Models consume plain packed tensors, so any loader that yields the [standard dict keys](../datasets/overview.md#dict-keys) works: there is no bespoke file format. This notebook writes a tiny dataset of `.npy` clouds, wraps it in a `Dataset`, collates it into packed batches, attaches a transform, and runs a model. Every cell runs on CPU.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import torch

root = Path(tempfile.mkdtemp()) / "my_clouds"
root.mkdir(parents=True)
for i in range(6):
    m = int(torch.randint(1500, 3000, (1,)))
    arr = np.concatenate([np.random.randn(m, 3), np.random.rand(m, 3)], axis=1).astype("float32")  # xyz + rgb
    np.save(root / f"cloud_{i:02d}.npy", arr)

sorted(p.name for p in root.glob("*.npy"))

## The standard keys

Return whichever of these your data has. Names follow the [Pointcept](https://github.com/Pointcept/Pointcept) convention; only `pos` is required.

| key        | shape    | meaning                         |
| ---------- | -------- | ------------------------------- |
| `pos`      | $(N, 3)$ | XYZ coordinates                 |
| `color`    | $(N, 3)$ | RGB (`[0, 255]` or `[0, 1]`)    |
| `normal`   | $(N, 3)$ | surface normals                 |
| `segment`  | $(N,)$   | per-point semantic labels       |
| `label`    | scalar   | one class for the whole cloud   |

## 1. A custom `Dataset`

A `torch.utils.data.Dataset` whose `__getitem__` returns the dict is all you need. Pass an optional `transform` so preprocessing travels with the dataset.

In [ ]:
from torch.utils.data import DataLoader, Dataset


class MyClouds(Dataset):
    def __init__(self, root, transform=None):
        self.files = sorted(Path(root).glob("*.npy"))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        arr = torch.from_numpy(np.load(self.files[i]))
        data = {"pos": arr[:, :3], "color": arr[:, 3:6]}
        return self.transform(data) if self.transform is not None else data


dataset = MyClouds(root)
{k: tuple(v.shape) for k, v in dataset[0].items()}

## 2. Collate into a packed batch

`collate` concatenates the per-point tensors along axis 0 and builds the `batch` index that tags each point with its source cloud. Scene-level tensors (a scalar `label`) are stacked instead.

In [ ]:
from torch_pointcloud.utils.data import collate

batch = collate([dataset[0], dataset[1], dataset[2]])
print({k: tuple(v.shape) for k, v in batch.items()})
print("clouds in batch:", int(batch["batch"].max()) + 1)
print("points from cloud 1:", int((batch["batch"] == 1).sum()))  # recover one cloud by masking

## 3. A DataLoader

Pass `collate` as `collate_fn` and the standard `DataLoader` does the rest: shuffling, workers, batching.

In [ ]:
loader = DataLoader(dataset, batch_size=3, shuffle=True, collate_fn=collate)
for step, batch in enumerate(loader):
    print(f"step {step}: pos {tuple(batch['pos'].shape)}, clouds {int(batch['batch'].max()) + 1}")

## 4. Attach a transform

The same [transforms](03-transforms.md) you compose by hand can ride along inside the dataset, applied per sample before collation. Here every cloud is centered and subsampled to a fixed 1024 points, so batches are uniform.

In [ ]:
import torch_pointcloud.transforms as T

pipeline = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSample(keys=("pos", "color"), num_samples=1024),
])
dataset = MyClouds(root, transform=pipeline)
{k: tuple(v.shape) for k, v in dataset[0].items()}

## 5. Feed a model

Nothing special here: a packed batch goes straight into `model(x, pos, batch)`. We build the architecture without weights so the cell runs offline.

In [ ]:
import torch_pointcloud as tp

model = tp.create_model("pointnet2-yanx27-ssg.modelnet40", task="classification").eval()
loader = DataLoader(dataset, batch_size=3, collate_fn=collate)
for batch in loader:
    with torch.no_grad():
        logits = model(None, batch["pos"], batch["batch"])
    print("logits:", tuple(logits.shape))  # (3, 40)
    break

## Tips

- **Dtypes.** `pos`, `color`, `normal` are float; `segment` / `label` are `long`.
- **Colour scale.** Some checkpoints expect `[0, 255]`, others `[0, 1]`; match the checkpoint's pipeline (`return_info=True`).
- **Meshes.** Store triangles under `face` $(F, 3)$ and sample points with `RandomSampleFaceVertices` (this is how `ModelNet40` turns CAD models into clouds).
- **Abstraction.** Library datasets subclass `PointCloudDataset`, which gives you `root` / `data_dir` / a nice `repr`. You still implement `__len__` and `__getitem__`:

```python
from torch_pointcloud.datasets.pointcloud import PointCloudDataset

class MyClouds(PointCloudDataset):
    def __init__(self, root, transform=None):
        super().__init__(root)
        self.files = sorted(Path(self.raw_dir).glob("*.npy"))
        self.transform = transform
    ...
```

See the [Datasets overview](../datasets/overview.md) for the built-in loaders, then [train a model](05-training.md) on your data.